In [7]:
import ee

ee.Initialize(project="ee-wildfire-recovery-vasant")

print("Earth Engine initialized")

Earth Engine initialized


### Retrieve Highline fire

In [8]:
mtbs = ee.FeatureCollection(
    "USFS/GTAC/MTBS/burned_area_boundaries/v1"
)

highline = mtbs.filter(
    ee.Filter.eq("Event_ID", "ID4549011522720170729")
).first()

print("Fire:", highline.get("Incid_Name").getInfo())
print("Acres:", highline.get("BurnBndAc").getInfo())

Fire: HIGHLINE
Acres: 93699


## Landsat Season 1 Scene Selection

For Highline, filtering `LANDSAT/LC08/C02/T1_L2` to the fire boundary and **June 1–September 30, 2018** returned **23 Landsat 8 scenes**. A scene is one satellite image acquisition covering a large geographic area.

We use **June 1–September 30** as our standardized post-fire growing-season window. This captures the main vegetation-active part of the year while avoiding much of the winter/snow period, and we will apply the same seasonal window consistently across fires and years. Because Highline burned in **2017**, **2018 is its first complete post-fire growing season (Season 1)**.

Because `filterBounds()` selects any scene that **intersects** Highline, a scene is not guaranteed to contain the entire fire. Some scenes may cover the whole perimeter, while others may cover only part of it.

Even within the part of Highline covered by a scene, **not every 30 m pixel is necessarily usable**. Individual pixels may be affected by clouds, cloud shadows, snow, saturation, or other quality problems.

The dataset choice already gives us:

- **Landsat 8**
- **Collection 2**
- **Tier 1**
- **Level 2 surface-reflectance imagery**

This query then further limits the collection to:

- the **Highline fire area**
- the **Season 1 date window**

### Get Highline's Landsat 8 images for first post fire growing season (2018-06-01 to 2018-10-01)

In [9]:
# "LANDSAT/LC08/C02/T1_L2"
# LC08 - Highline burned in 2017, and all of our POC periods (2014–16 pre-fire, 2018 Season 1, 2022 Season 5) are within Landsat 8's operating period
# C02 - This is USGS's newer reprocessed Landsat collection with updated calibration/geometric processing
# Tier 1 - Highest-quality scenes suitable for time-series analysis, with strong geometric/geolocation quality
# Level 2 - Gives us atmospherically corrected surface reflectance, which is what we want for vegetation indices like NBR
landsat_s1 = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(highline.geometry())
    .filterDate("2018-06-01", "2018-10-01")
)

print(
    "Number of Landsat 8 scenes intersecting Highline:",
    landsat_s1.size().getInfo()
)

Number of Landsat 8 scenes intersecting Highline: 23


The **23 scenes are therefore candidate observations**, not 23 fully usable measurements for every Highline pixel.

We still need to:

1. Apply the surface-reflectance scaling.
2. Mask bad pixels using Landsat QA information.
3. Calculate NBR for every valid pixel observation.
4. Combine the valid June–September observations into one seasonal NBR value per **30 m pixel**.

Later, those 30 m seasonal NBR values will be aggregated into our **90 × 90 m modeling patches**.

## Step 2 — Inspecting Highline's Season 1 Landsat Scenes

After filtering the Landsat 8 Collection 2 Tier 1 Level 2 archive to:

- the **Highline fire perimeter**
- **June 1–September 30, 2018**

we found **23 candidate Landsat scenes**.

### What is a Landsat scene?

Landsat continuously collects imagery as the satellite moves along its orbit. For distribution and analysis, this imagery is divided into standardized geographic units called **scenes**.

A typical Landsat scene covers roughly **185 km × 180 km**, which is much larger than the Highline fire.

Therefore, Highline occupies only a portion of each Landsat scene.

Importantly, a scene that intersects Highline does **not necessarily cover the entire fire**. Some scenes may contain all of Highline, while others may contain only part of the fire because Highline lies near the edge of that scene's footprint.

### What are Path and Row?

Landsat organizes the Earth's surface using the **Worldwide Reference System (WRS-2)**.

Each standard Landsat scene is identified by a combination of:

- **Path** — identifies the satellite's orbital track across the Earth.
- **Row** — identifies a section along that orbital track.

Together, the path and row identify the approximate geographic location of a Landsat scene.

Conceptually:

`Path = which orbital strip`

`Row = which section along that strip`

`Path + Row = one standardized Landsat scene location`

In [10]:
# Summarize the candidate Landsat scenes for Highline Season 1

def scene_metadata(image):
    return ee.Feature(
        None,
        {
            "date": ee.Date(image.get("system:time_start")).format("YYYY-MM-dd"),
            "path": image.get("WRS_PATH"),
            "row": image.get("WRS_ROW"),
            "cloud_cover": image.get("CLOUD_COVER"),
        },
    )

scene_table = ee.FeatureCollection(landsat_s1.map(scene_metadata))

scene_info = scene_table.getInfo()["features"]

for scene in scene_info:
    props = scene["properties"]

    print(
        props["date"],
        "| Path:", props["path"],
        "| Row:", props["row"],
        "| Scene cloud %:", round(props["cloud_cover"], 1),
    )

2018-06-12 | Path: 41 | Row: 28 | Scene cloud %: 3.1
2018-06-28 | Path: 41 | Row: 28 | Scene cloud %: 54.8
2018-07-14 | Path: 41 | Row: 28 | Scene cloud %: 0.5
2018-07-30 | Path: 41 | Row: 28 | Scene cloud %: 0.1
2018-08-15 | Path: 41 | Row: 28 | Scene cloud %: 0.1
2018-08-31 | Path: 41 | Row: 28 | Scene cloud %: 31.8
2018-09-16 | Path: 41 | Row: 28 | Scene cloud %: 14.6
2018-06-12 | Path: 41 | Row: 29 | Scene cloud %: 1.8
2018-06-28 | Path: 41 | Row: 29 | Scene cloud %: 60.8
2018-07-14 | Path: 41 | Row: 29 | Scene cloud %: 0.5
2018-07-30 | Path: 41 | Row: 29 | Scene cloud %: 0.1
2018-08-15 | Path: 41 | Row: 29 | Scene cloud %: 0.0
2018-08-31 | Path: 41 | Row: 29 | Scene cloud %: 7.9
2018-09-16 | Path: 41 | Row: 29 | Scene cloud %: 0.3
2018-06-03 | Path: 42 | Row: 28 | Scene cloud %: 5.0
2018-06-19 | Path: 42 | Row: 28 | Scene cloud %: 84.8
2018-07-05 | Path: 42 | Row: 28 | Scene cloud %: 0.5
2018-07-21 | Path: 42 | Row: 28 | Scene cloud %: 7.3
2018-08-06 | Path: 42 | Row: 28 | Scene c

In [11]:
first_image = ee.Image(landsat_s1.first())

print(first_image.bandNames().getInfo())

['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT']


For Highline, the Season 1 results included:

- Paths **41 and 42**
- Rows **28 and 29**

This means Highline lies close enough to Landsat scene boundaries that it intersects multiple neighboring scene footprints.

### Why do some dates appear more than once?

For example, we have observations such as:

`2018-06-12 | Path 41 | Row 28`

and

`2018-06-12 | Path 41 | Row 29`

These are neighboring scene sections from the **same satellite pass on the same date**. Highline intersects both row footprints, so both scenes are returned by `filterBounds()`.

This does **not** mean Landsat independently photographed Highline twice that day. It means the same orbital acquisition is divided into neighboring standard scene units that both intersect our fire.

### How often does Landsat 8 observe an area?

Landsat 8 has a **16-day repeat cycle for the same orbital path**.

We can see this directly in our results for Path 41:

- June 12
- June 28
- July 14
- July 30
- August 15
- August 31
- September 16

Each observation is approximately **16 days apart**.

However, Highline also intersects neighboring **Path 42**. Path 42 passes over the area on different dates:

- June 3
- June 19
- July 5
- July 21
- August 6
- August 22
- September 7
- September 23

Because Highline is visible from two overlapping Landsat paths, it can receive a potential observation roughly every **7–9 days**, even though Landsat 8 returns to the exact same path only every 16 days.

This extra overlap is useful because it gives us more potential observations during the growing season.

### Scene-level cloud cover

We also inspected the `CLOUD_COVER` metadata for every scene.

Cloud cover ranged from almost **0%** to more than **80%**.

However, this percentage describes cloudiness across the **entire Landsat scene**, not specifically across Highline.

For example:

- A scene could be 80% cloudy overall but have Highline located in the clear 20%.
- A scene could be only 5% cloudy overall but have that small cloud sitting directly over part of Highline.

Therefore, we will **not primarily discard entire scenes based on scene-level cloud percentage**.

Instead, we will use Landsat's **pixel-level QA information** so that usable 30 m pixels can be retained while cloudy or otherwise unreliable pixels from the same scene are removed.

### Important distinction: scene vs. usable pixel observation

The 23 scenes are therefore **23 candidate image products**, not 23 valid observations for every point inside Highline.

For one particular 30 m location, the situation might look like:

- Scene 1 → clear → usable
- Scene 2 → cloud → unusable
- Scene 3 → clear → usable
- Scene 4 → cloud shadow → unusable
- Scene 5 → clear → usable

Another 30 m location within Highline may have a different set of valid observations.

This is why our later seasonal NBR calculation is performed **pixel-by-pixel using only valid observations at each location**.

### Bands confirmed

We also confirmed that the selected Landsat dataset contains the bands required for our NBR workflow:

- `SR_B5` — Near Infrared (NIR)
- `SR_B7` — Shortwave Infrared 2 (SWIR2)
- `QA_PIXEL` — pixel-level cloud, shadow, snow, and other quality information
- `QA_RADSAT` — radiometric saturation and related quality information


### What Does It Mean That a Landsat Scene Is a Raster?

A Landsat scene is a **raster**, meaning the geographic area is represented as a grid of pixels. For Landsat's optical data, each pixel represents approximately **30 m × 30 m** on the ground.

The scene contains multiple aligned **bands** (layers). Each band stores a different measurement for the same grid of geographic locations.

For example, one 30 m pixel might have:

- Blue reflectance
- Red reflectance
- NIR reflectance
- SWIR1 reflectance
- SWIR2 reflectance
- Pixel-quality information
- Saturation-quality information

The neighboring 30 m pixel has its own separate values for all of those same bands.

Therefore, a Landsat scene can be thought of as a **stack of raster layers describing millions of individual 30 m locations on one acquisition date**.

## Step 3 — Landsat Quality Masking and Surface-Reflectance Scaling

Before calculating NBR, the Landsat measurements must be cleaned so that only trustworthy pixel observations are used.

For each Landsat scene, the preprocessing function performs two main tasks:

1. **Mask unreliable 30 m pixel observations**
2. **Convert the stored Landsat values into actual surface reflectance**

### Pixel-Level Quality Control

Landsat provides a `QA_PIXEL` band containing quality information for every 30 m pixel.

Relevant flags include:

- Fill / no valid image data
- Dilated cloud
- Cirrus cloud
- Cloud
- Cloud shadow
- Snow / ice

These conditions are stored as binary **bits** inside the `QA_PIXEL` value.

For each pixel, we check whether any of these unwanted flags are present.

If none are present, the observation is kept.

If any are present, the observation is **masked**, meaning it is treated as invalid/missing and will be ignored in later calculations rather than assigned a value of zero.

### Radiometric Saturation and Terrain Quality

We also use the `QA_RADSAT` band.

This indicates whether individual spectral bands became **radiometrically saturated**, meaning the sensor reached the limit of what it could accurately measure.

Because NBR uses:

- `SR_B5` — Near Infrared (NIR)
- `SR_B7` — Shortwave Infrared 2 (SWIR2)

we specifically check whether Bands 5 or 7 are saturated.

We also remove pixels flagged for **terrain occlusion**, where terrain geometry may have prevented a reliable observation.

A pixel is therefore retained only if it passes all of the relevant quality checks.

### Why Only `SR_B5` and `SR_B7`?

The Landsat Level 2 product contains many bands because it supports many different remote-sensing applications.

For the NBR calculation, we specifically need:

- `SR_B5` → Near Infrared (NIR)
- `SR_B7` → Shortwave Infrared 2 (SWIR2)

Other surface-reflectance bands are not required for NBR, although some may become useful later for additional features such as NDVI or NDMI.

The `ST_*` bands relate to Landsat surface-temperature products and are not needed for the current NBR workflow.

`SR_QA_AEROSOL` contains aerosol-quality information. It is not included in the current POC mask, but whether high-aerosol or unreliable aerosol-retrieval pixels should also be removed will be audited before the production Landsat QA policy is finalized.

### Surface-Reflectance Scaling

The Landsat surface-reflectance bands are stored as scaled integer **Digital Numbers (DNs)** rather than directly as decimal reflectance values.

For Landsat Collection 2 Level 2 surface reflectance, the conversion is:

`Reflectance = DN × 0.0000275 - 0.2`

For example:

`DN = 20000`

becomes:

`Reflectance = 0.35`

The preprocessing function therefore converts `SR_B5` and `SR_B7` into actual surface-reflectance values and renames them:

- `SR_B5` → `NIR`
- `SR_B7` → `SWIR2`

### What the Preprocessing Function Produces

For one Landsat scene, the conceptual transformation is:

`Original Landsat scene`

→ check `QA_PIXEL`

→ check `QA_RADSAT`

→ mask bad observations

→ scale `SR_B5` and `SR_B7`

→ output clean `NIR` and `SWIR2` reflectance bands

For example:

- Pixel A → clear → retain NIR and SWIR2
- Pixel B → cloud → masked
- Pixel C → clear → retain NIR and SWIR2
- Pixel D → cloud shadow → masked

In [ ]:
def preprocess_landsat8(image):
    """
    Prepare one Landsat 8 Collection 2 Level 2 scene for NBR.
    """

    # ---------------------------------------------------------
    # 1. Mask bad pixels using QA_PIXEL
    # ---------------------------------------------------------
    qa_pixel = image.select("QA_PIXEL")

    # Bits 0-5:
    # 0 = fill
    # 1 = dilated cloud
    # 2 = cirrus
    # 3 = cloud
    # 4 = cloud shadow
    # 5 = snow
    bad_qa_bits = (
        (1 << 0)
        | (1 << 1)
        | (1 << 2)
        | (1 << 3)
        | (1 << 4)
        | (1 << 5)
    )

    qa_mask = qa_pixel.bitwiseAnd(bad_qa_bits).eq(0)

    # ---------------------------------------------------------
    # 2. Mask saturation / terrain problems
    # ---------------------------------------------------------
    qa_radsat = image.select("QA_RADSAT")

    nir_ok = qa_radsat.bitwiseAnd(1 << 4).eq(0)
    swir2_ok = qa_radsat.bitwiseAnd(1 << 6).eq(0)
    terrain_ok = qa_radsat.bitwiseAnd(1 << 11).eq(0)

    quality_mask = (
        qa_mask
        .And(nir_ok)
        .And(swir2_ok)
        .And(terrain_ok)
    )

    # ---------------------------------------------------------
    # 3. Convert stored DN values to surface reflectance
    # ---------------------------------------------------------
    reflectance = (
        image.select(["SR_B5", "SR_B7"])
        .multiply(0.0000275)
        .add(-0.2)
        .rename(["NIR", "SWIR2"])
    )

    processed = (
        reflectance
        .updateMask(quality_mask)
        .copyProperties(image, image.propertyNames())
    )

    return ee.Image(processed)

In [15]:
first_scene = ee.Image(landsat_s1.first())

processed_first_scene = preprocess_landsat8(first_scene)

print(
    "Scene date:",
    ee.Date(first_scene.get("system:time_start"))
    .format("YYYY-MM-dd")
    .getInfo()
)

print(
    "Processed bands:",
    processed_first_scene.bandNames().getInfo()
)

Scene date: 2018-06-12
Processed bands: ['NIR', 'SWIR2']


In [16]:
reflectance_stats = processed_first_scene.reduceRegion(
    reducer=ee.Reducer.percentile([1, 50, 99]),
    geometry=highline.geometry(),
    scale=30,
    bestEffort=True,
    maxPixels=1e8,
)

print(reflectance_stats.getInfo())

{'NIR_p1': 0.04117672127355311, 'NIR_p50': 0.13084871105603096, 'NIR_p99': 0.2792483573851247, 'SWIR2_p1': 0.04788144994864616, 'SWIR2_p50': 0.13966093295712415, 'SWIR2_p99': 0.23532526536555703}


### Testing on One Scene

The preprocessing function was first tested on a single Season 1 Landsat scene from **June 12, 2018** rather than immediately applying it to all 23 candidate scenes.

This is useful because it allows the preprocessing logic to be checked on one real image before scaling it to the full collection.

The processed scene correctly contained only:

- `NIR`
- `SWIR2`

### Reflectance Sanity Check

To verify that the scaling worked, the 1st, 50th, and 99th percentiles of NIR and SWIR2 were calculated for valid pixels within the Highline fire perimeter for this one scene.

Results:

- NIR 1st percentile: ~0.041
- NIR median: ~0.131
- NIR 99th percentile: ~0.279
- SWIR2 1st percentile: ~0.048
- SWIR2 median: ~0.140
- SWIR2 99th percentile: ~0.235

These values are on a plausible surface-reflectance scale, confirming that the stored Landsat integer values were successfully converted.

These percentiles summarize the **spatial distribution of valid Highline pixels in this one June 12 scene only**. They are not seasonal values and are not `NBR_1`.

### Earth Engine Computation Scope

The preprocessing function is defined for the Landsat scene, but Earth Engine uses **lazy evaluation**.

This means the full scene is not immediately processed on the local computer.

When the percentile calculation was requested using:

`geometry = highline.geometry()`

Earth Engine evaluated only the pixels needed within the Highline fire perimeter.

Also:

- `filterBounds()` selects scenes that intersect Highline.
- It does not crop the scenes to the fire boundary.
- Geographic restrictions are applied when reductions, exports, or other computations specify the Highline geometry.

### Where We Are After Step 3

At this point we have:

- identified the relevant Landsat NIR and SWIR2 bands,
- created pixel-level quality masks,
- removed cloud, shadow, snow, saturation, and terrain-related problems,
- converted stored values to surface reflectance,
- tested the preprocessing workflow on one real scene,
- confirmed that the resulting reflectance values are plausible.

We have **not yet calculated NBR** and we have **not yet combined multiple dates into a seasonal median**.

The next step is to calculate:

`NBR = (NIR - SWIR2) / (NIR + SWIR2)`

for every valid pixel observation.

## Step 4 — Calculating NBR for Each Valid Landsat Observation

After Step 3, each Landsat scene has been reduced to two cleaned and scaled surface-reflectance bands:

- `NIR`
- `SWIR2`

Pixels affected by clouds, cloud shadows, snow, saturation, terrain occlusion, or other major quality problems have already been masked.

Step 4 converts those two reflectance values into **Normalized Burn Ratio (NBR)** for every valid 30 m pixel observation.

### NBR Formula

NBR is calculated as:

`NBR = (NIR - SWIR2) / (NIR + SWIR2)`

For each valid pixel in a scene:

1. Read the pixel's NIR reflectance.
2. Read the pixel's SWIR2 reflectance.
3. Calculate the NBR value using the formula above.
4. Mask the pixel if the denominator is zero.
5. Keep the original scene metadata such as acquisition date, path, and row.

The output for each Landsat scene is therefore a **single-band NBR raster**.

Conceptually:

`Clean NIR raster + Clean SWIR2 raster`

→ calculate NBR independently for each valid 30 m pixel

→ produce one NBR raster for that acquisition

For example:

- Pixel A → valid NIR/SWIR2 → calculate NBR
- Pixel B → previously masked → remains masked
- Pixel C → valid NIR/SWIR2 → calculate NBR

### Applying the Function to the Full Season 1 Collection

The NBR function was applied to all **23 candidate Landsat scenes** from Highline's 2018 Season 1 window.

This produced:

**23 scene-level NBR rasters**

Each raster corresponds to one Landsat scene acquisition and contains an NBR value for every valid 30 m pixel covered by that scene.

At this stage, these are still **individual-date observations**. They have not yet been combined into the final Season 1 NBR raster.

In [17]:
def calculate_nbr(image):
    """
    Calculate NBR from a preprocessed Landsat image
    containing NIR and SWIR2 surface reflectance.
    """

    nir = image.select("NIR")
    swir2 = image.select("SWIR2")

    denominator = nir.add(swir2)

    nbr = (
        nir.subtract(swir2)
        .divide(denominator)
        .rename("NBR")
        .updateMask(denominator.neq(0))
    )

    # Keep the original scene/date metadata
    return ee.Image(
        nbr.copyProperties(image, image.propertyNames())
    )

In [18]:
landsat_s1_processed = landsat_s1.map(preprocess_landsat8)

landsat_s1_nbr = landsat_s1_processed.map(calculate_nbr)

print("Number of NBR scenes:", landsat_s1_nbr.size().getInfo())

first_nbr_scene = ee.Image(landsat_s1_nbr.first())

print(
    "First NBR scene date:",
    ee.Date(first_nbr_scene.get("system:time_start"))
    .format("YYYY-MM-dd")
    .getInfo()
)

print(
    "Bands:",
    first_nbr_scene.bandNames().getInfo()
)

Number of NBR scenes: 23
First NBR scene date: 2018-06-12
Bands: ['NBR']


In [19]:
nbr_stats = first_nbr_scene.reduceRegion(
    reducer=ee.Reducer.percentile([1, 50, 99]),
    geometry=highline.geometry(),
    scale=30,
    bestEffort=True,
    maxPixels=1e8,
)

print(nbr_stats.getInfo())

{'NBR_p1': -0.5038341446388896, 'NBR_p50': -0.04298690258687666, 'NBR_p99': 0.550725574149077}


In [20]:
def summarize_nbr_scene(image):
    stats = image.reduceRegion(
        reducer=(
            ee.Reducer.percentile([1, 50, 99])
            .combine(
                reducer2=ee.Reducer.count(),
                sharedInputs=True
            )
        ),
        geometry=highline.geometry(),
        scale=30,
        bestEffort=True,
        maxPixels=1e8,
    )

    return ee.Feature(
        None,
        stats
    ).set({
        "date": ee.Date(image.get("system:time_start")).format("YYYY-MM-dd"),
        "path": image.get("WRS_PATH"),
        "row": image.get("WRS_ROW"),
        "cloud_cover": image.get("CLOUD_COVER"),
    })


nbr_scene_summaries = ee.FeatureCollection(
    landsat_s1_nbr.map(summarize_nbr_scene)
)

summary_info = nbr_scene_summaries.getInfo()["features"]

for feature in summary_info:
    p = feature["properties"]

    print(
        p["date"],
        "| Path:", p["path"],
        "| Row:", p["row"],
        "| Cloud %:", round(p["cloud_cover"], 1),
        "| Valid pixels:", p.get("NBR_count"),
        "| NBR p1:", round(p["NBR_p1"], 3) if p.get("NBR_p1") is not None else None,
        "| p50:", round(p["NBR_p50"], 3) if p.get("NBR_p50") is not None else None,
        "| p99:", round(p["NBR_p99"], 3) if p.get("NBR_p99") is not None else None,
    )

2018-06-12 | Path: 41 | Row: 28 | Cloud %: 3.1 | Valid pixels: 424836 | NBR p1: -0.504 | p50: -0.043 | p99: 0.551
2018-06-28 | Path: 41 | Row: 28 | Cloud %: 54.8 | Valid pixels: 0 | NBR p1: None | p50: None | p99: None
2018-07-14 | Path: 41 | Row: 28 | Cloud %: 0.5 | Valid pixels: 426109 | NBR p1: -0.473 | p50: -0.012 | p99: 0.574
2018-07-30 | Path: 41 | Row: 28 | Cloud %: 0.1 | Valid pixels: 427231 | NBR p1: -0.457 | p50: -0.004 | p99: 0.574
2018-08-15 | Path: 41 | Row: 28 | Cloud %: 0.1 | Valid pixels: 427233 | NBR p1: -0.441 | p50: 0.004 | p99: 0.559
2018-08-31 | Path: 41 | Row: 28 | Cloud %: 31.8 | Valid pixels: 285224 | NBR p1: -0.421 | p50: 0.061 | p99: 0.549
2018-09-16 | Path: 41 | Row: 28 | Cloud %: 14.6 | Valid pixels: 427192 | NBR p1: -0.449 | p50: 0.02 | p99: 0.543
2018-06-12 | Path: 41 | Row: 29 | Cloud %: 1.8 | Valid pixels: 423602 | NBR p1: -0.504 | p50: -0.043 | p99: 0.551
2018-06-28 | Path: 41 | Row: 29 | Cloud %: 60.8 | Valid pixels: 0 | NBR p1: None | p50: None | p99:

## Collection-Level Sanity Check

Rather than inspecting only one NBR raster, every one of the 23 scene-level NBR rasters was summarized over the Highline fire perimeter.

For each scene, we examined:

- acquisition date
- WRS path
- WRS row
- scene-level cloud cover
- number of valid Highline pixels after masking
- NBR 1st percentile
- NBR median
- NBR 99th percentile

This allowed us to check whether the NBR calculation behaved sensibly across the full collection before creating a seasonal composite.

### NBR Values Look Plausible

Most usable scenes produced NBR distributions roughly within:

- 1st percentile: approximately `-0.5` to `-0.2`
- median: approximately `-0.05` to `0.19`
- 99th percentile: approximately `0.5` to `0.66`

These values are within the expected approximate NBR range of `-1` to `+1`.

The presence of negative, near-zero, and strongly positive values is plausible for a heterogeneous landscape recovering from wildfire, where individual pixels may represent burned ground, sparse vegetation, recovering vegetation, or relatively intact forest.

No extreme values indicating an obvious scaling or formula error were observed.

### Some Scenes Have Zero Valid Highline Pixels

Several scenes produced zero valid pixels inside Highline after QA masking.

Examples included highly cloudy scenes such as:

- June 28, 2018
- June 19, 2018

This does not necessarily indicate an error.

A Landsat scene can intersect Highline but still contribute no usable observations if:

- the Highline portion of the scene is cloudy or otherwise masked, or
- only a small portion of the fire lies inside that particular scene footprint and all of those pixels fail the quality checks.

Scenes with zero valid observations will simply contribute nothing to the later seasonal composite.

### Valid-Pixel Counts Reflect Scene Coverage

Some scenes contained approximately **420,000–427,000 valid Highline pixels**, while others contained far fewer.

This makes geographic sense.

A 30 m Landsat pixel covers:

`30 m × 30 m = 900 m²`

Highline covers approximately 93,700 acres, so roughly several hundred thousand 30 m pixels are expected across the full fire perimeter.

Scenes with approximately 420,000 valid pixels therefore appear to cover nearly the entire fire.

Scenes with much smaller counts, such as approximately 150,000 pixels, likely cover only part of Highline because the fire intersects the edge of that Landsat scene footprint.

### Important Same-Date Overlap Issue

The sanity check also revealed an important compositing issue.

Highline intersects neighboring WRS rows, so the collection includes examples such as:

`2018-06-12 | Path 41 | Row 28`

and

`2018-06-12 | Path 41 | Row 29`

These are not two independent satellite observations. They are neighboring scene products from the **same Landsat pass on the same date**.

Their Highline NBR statistics are often nearly identical, confirming that they contain substantial overlapping information.

Therefore, we should **not blindly treat all 23 scene objects as 23 independent temporal observations** when calculating the seasonal median.

Before temporal compositing, same-acquisition overlapping scene products must be handled appropriately so that one physical satellite pass is not accidentally given extra weight.

This will be addressed during the seasonal-composite step.

### One Scene Worth Further Inspection

The August 31, 2018 Path 41 Row 29 scene showed an unusual result where its median and 99th-percentile NBR values were both approximately `0.194`.

This does not automatically indicate an error, but it is unusual enough to keep under observation when we build and inspect the final seasonal composite.

### Status After Step 4

At the end of Step 4:

- all 23 Season 1 Landsat scenes have been preprocessed,
- NBR has been calculated for every valid 30 m pixel observation,
- we now have 23 scene-level NBR rasters,
- the resulting NBR values appear physically plausible,
- scenes with poor usable coverage have been identified,
- and the same-date neighboring-scene overlap issue has been identified before temporal compositing.

We still have **not created `NBR_1`**.

The next step is to correctly combine the valid scene-level NBR observations through time into a single **June–September 2018 seasonal NBR raster**, with one seasonal NBR value per 30 m location.

## Step 5 — Building the Season 1 NBR Composite

After Step 4, we had **23 scene-level NBR rasters** for Highline's first complete post-fire growing season:

**June 1–September 30, 2018**

Each raster represented NBR values for one Landsat scene acquisition, with bad pixels already masked.

The goal of Step 5 was to convert those multiple scene-level observations into:

- one **Season 1 NBR raster (`NBR_1`)**
- one **valid-observation-count raster**

### Step 5A — Combine Scene Products From the Same Date

Some Landsat acquisitions were represented by multiple neighboring scene products.

For example:

`2018-06-12 | Path 41 | Row 28`

and

`2018-06-12 | Path 41 | Row 29`

These are separate scene products, but they correspond to imagery collected on the same calendar date and overlap spatially over parts of Highline.

If we treated both scene products as separate temporal observations, the same date could receive extra weight in the seasonal median.

To avoid this, all NBR scene products from the same date were combined into a single **daily NBR raster** using a pixel-wise median.

For each geographic pixel:

- only Scene A has a valid value → use Scene A
- only Scene B has a valid value → use Scene B
- both have valid values → take the median of the two
- neither has a valid value → the output pixel remains masked

Masked or missing pixels are **not treated as zero**.

This reduced:

**23 scene products → 15 unique acquisition dates**

The 15 resulting daily NBR rasters represent the usable Landsat observations through the 2018 growing season without allowing multiple same-date scene products to count as separate temporal observations.

### Step 5B — Calculate the Seasonal NBR Raster

Once there was one NBR raster per acquisition date, the 15 daily rasters were combined through time.

For every fixed 30 m geographic location, Earth Engine collected all valid NBR observations available between June and September and calculated their median.

Conceptually:

`June NBR`

`July NBR`

`August NBR`

`September NBR`

→ ignore masked observations

→ take the median of the valid values

→ one Season 1 NBR value for that 30 m location

This is done independently for every 30 m pixel.

The resulting raster was renamed:

`NBR_1`

Therefore, `NBR_1` is a spatial raster in which each valid 30 m location contains one robust NBR value summarizing its first complete post-fire growing season.

### Why Use a Median?

A median is more resistant than a mean to occasional residual outliers or imperfect observations that survive QA masking.

For example:

`0.20, 0.22, 0.23, 0.21, 0.24, 0.85`

The unusually high `0.85` value would pull the mean upward, while the median would remain much closer to the typical seasonal condition.

The seasonal median is therefore being used as a robust summary of vegetation spectral condition during the growing season.

### Step 5C — Track Valid Observation Counts

A seasonal median based on many observations is more trustworthy than one based on only one or two.

Therefore, a second raster was created:

`valid_obs_count`

For every 30 m pixel, this raster stores the number of acquisition dates that contributed a valid NBR measurement to the seasonal median.

For example:

- Pixel A → 12 valid dates
- Pixel B → 6 valid dates
- Pixel C → 4 valid dates

The observation-count raster has the same geography as `NBR_1`, but its values represent **data availability rather than vegetation condition**.

This count can later be used for data-quality filtering and coverage analysis.



In [24]:
# Get the unique acquisition dates in Season 1
season1_dates = (
    ee.List(landsat_s1_nbr.aggregate_array("system:time_start"))
    .map(lambda t: ee.Date(t).format("YYYY-MM-dd"))
    .distinct()
    .sort()
)


def combine_acquisition_date(date_string):
    """
    Combine all Landsat scene pieces from the same acquisition date
    into one NBR image.
    """

    date = ee.Date.parse("YYYY-MM-dd", ee.String(date_string))

    same_day_scenes = landsat_s1_nbr.filterDate(
        date,
        date.advance(1, "day")
    )

    daily_nbr = (
        same_day_scenes
        .median()
        .rename("NBR")
    )

    return ee.Image(daily_nbr).set({
        "date": date.format("YYYY-MM-dd"),
        "system:time_start": date.millis(),
        "source_scene_count": same_day_scenes.size(),
    })


landsat_s1_daily_nbr = ee.ImageCollection.fromImages(
    season1_dates.map(combine_acquisition_date)
)

print("Original scene products:", landsat_s1_nbr.size().getInfo())
print("Unique acquisition dates:", landsat_s1_daily_nbr.size().getInfo())

print("\nAcquisitions:")

daily_info = landsat_s1_daily_nbr.getInfo()["features"]

for feature in daily_info:
    p = feature["properties"]

    print(
        p["date"],
        "| Source scene products:",
        p["source_scene_count"]
    )

Original scene products: 23
Unique acquisition dates: 15

Acquisitions:
2018-06-03 | Source scene products: 1
2018-06-12 | Source scene products: 2
2018-06-19 | Source scene products: 1
2018-06-28 | Source scene products: 2
2018-07-05 | Source scene products: 1
2018-07-14 | Source scene products: 2
2018-07-21 | Source scene products: 2
2018-07-30 | Source scene products: 2
2018-08-06 | Source scene products: 1
2018-08-15 | Source scene products: 2
2018-08-22 | Source scene products: 1
2018-08-31 | Source scene products: 2
2018-09-07 | Source scene products: 1
2018-09-16 | Source scene products: 2
2018-09-23 | Source scene products: 1


In [25]:
# Pixel-wise median NBR across all unique acquisition dates
nbr_s1 = (
    landsat_s1_daily_nbr
    .median()
    .rename("NBR_1")
)

# Number of valid acquisition-date observations available
# for each 30 m pixel
nbr_s1_obs_count = (
    landsat_s1_daily_nbr
    .count()
    .rename("valid_obs_count")
)

print("Season 1 raster band:", nbr_s1.bandNames().getInfo())
print(
    "Observation-count band:",
    nbr_s1_obs_count.bandNames().getInfo()
)

Season 1 raster band: ['NBR_1']
Observation-count band: ['valid_obs_count']


In [26]:
seasonal_nbr_stats = nbr_s1.reduceRegion(
    reducer=ee.Reducer.percentile([1, 50, 99]),
    geometry=highline.geometry(),
    scale=30,
    bestEffort=True,
    maxPixels=1e8,
)

observation_count_stats = nbr_s1_obs_count.reduceRegion(
    reducer=(
        ee.Reducer.percentile([1, 10, 50, 90, 99])
        .combine(
            reducer2=ee.Reducer.minMax(),
            sharedInputs=True
        )
    ),
    geometry=highline.geometry(),
    scale=30,
    bestEffort=True,
    maxPixels=1e8,
)

print("Season 1 NBR statistics:")
print(seasonal_nbr_stats.getInfo())

print("\nValid observation-count statistics:")
print(observation_count_stats.getInfo())

Season 1 NBR statistics:
{'NBR_1_p1': -0.45696043107787226, 'NBR_1_p50': 0.003924816483028613, 'NBR_1_p99': 0.5507525969881503}

Valid observation-count statistics:
{'valid_obs_count_max': 13, 'valid_obs_count_min': 4, 'valid_obs_count_p1': 5.000000000000002, 'valid_obs_count_p10': 5.000000000000002, 'valid_obs_count_p50': 5.9999999999999725, 'valid_obs_count_p90': 11.000000000000009, 'valid_obs_count_p99': 12.00000000000002}


## Results

### Scene Products vs. Acquisition Dates

The collection contained:

- **23 original scene products**
- **15 unique acquisition dates**

This confirms that several dates contained multiple neighboring Landsat scene products that needed to be consolidated before temporal compositing.

### Season 1 NBR Distribution

The final `NBR_1` raster produced approximately:

- 1st percentile: `-0.457`
- median: `0.0039`
- 99th percentile: `0.551`

These values are physically plausible for a heterogeneous landscape during the first growing season after wildfire.

Importantly, the reported median of approximately `0.0039` is only a **spatial summary of the final raster for sanity checking**.

It does **not** mean that every Highline pixel receives an NBR value of `0.0039`.

Each 30 m location retains its own independent seasonal NBR value.

### Valid Observation Coverage

The `valid_obs_count` raster showed approximately:

- minimum: `4`
- 1st percentile: `5`
- 10th percentile: `5`
- median: `6`
- 90th percentile: `11`
- 99th percentile: `12`
- maximum: `13`

There were 15 possible acquisition dates during the Season 1 collection.

A typical Highline pixel therefore had approximately **6 valid NBR observations** contributing to its seasonal median.

Even the lowest-coverage pixels represented in this result had at least **4 valid observations**.

This is encouraging for the proof of concept, although a final minimum valid-observation threshold has **not yet been chosen**. That threshold will be determined empirically during later data-quality analysis.

---

## Interpreting NBR

NBR is calculated as:

`NBR = (NIR - SWIR2) / (NIR + SWIR2)`

Its usual conceptual range is approximately:

`-1 to +1`

A rough interpretation is:

- **Higher positive NBR** → NIR dominates SWIR2; often associated with denser, healthier, or moister vegetation.
- **NBR near zero** → NIR and SWIR2 are similar; can occur in sparse vegetation, exposed soil, burned areas, or mixed surfaces.
- **Negative NBR** → SWIR2 dominates NIR; often associated with more strongly burned, dry, bare, or otherwise non-vegetated spectral conditions.

NBR is a **dimensionless spectral index**, not a percentage of vegetation health.

For example, `NBR = 0.60` does **not** mean that vegetation is "60% healthy."

Instead, NBR summarizes the relative behavior of NIR and SWIR2 reflectance.

This makes it useful for tracking wildfire damage and vegetation recovery through time.

---

## Status After Step 5

The Season 1 Landsat workflow now performs:

`Landsat scenes`

→ QA masking

→ surface-reflectance scaling

→ NBR calculation

→ same-date spatial consolidation

→ pixel-wise temporal median

→ `NBR_1` seasonal raster

Alongside `NBR_1`, we also retain `valid_obs_count` to measure how much valid Landsat information supports each seasonal pixel value.

### Methodology Note for the Full Production Pipeline

For this Highline proof of concept, scene products are grouped by **calendar date** before seasonal compositing.

This is a coherent approach because it produces one spatial NBR representation per day and prevents multiple same-day scene products from receiving extra temporal weight.

However, before the full multi-fire Landsat pipeline is finalized, the grouping policy should be explicitly revisited, especially when combining multiple Landsat sensors such as Landsat 7, 8, and 9.

The production workflow should compare and document whether the desired temporal unit is:

- one observation per physical satellite acquisition / overpass, or
- one combined daily observation across all valid imagery acquired that date.

The effect of sensor harmonization and same-day multi-sensor observations should also be audited before this methodology is locked.

### Step 6.1 — Generalizing the Seasonal Landsat Workflow

After successfully building the 2018 Season 1 NBR composite, the same logic was converted into a reusable function:

`build_seasonal_nbr(year)`

For any requested year, the function:

1. Retrieves Landsat 8 Collection 2 Tier 1 Level 2 scenes intersecting Highline from **June 1 through September 30**.
2. Applies the previously defined Landsat preprocessing workflow:
   - pixel-level QA masking
   - saturation / terrain-quality checks
   - surface-reflectance scaling
3. Calculates NBR for every valid pixel observation.
4. Identifies unique acquisition dates.
5. Combines multiple scene products from the same calendar date using a pixel-wise median.
6. Takes the temporal median across valid dates to create one seasonal NBR raster.
7. Counts how many valid acquisition dates contributed to each 30 m location.

The function returns:

- the seasonal NBR raster
- a valid-observation-count raster
- the original Landsat scene count
- the number of unique acquisition dates

This lets the same processing methodology be applied consistently across all years instead of manually rebuilding the workflow each time.

In [ ]:
def build_seasonal_nbr(year):
    """
    Build a June 1–September 30 seasonal NBR composite for Highline
    for a given year.

    Returns:
        - seasonal NBR raster
        - valid observation-count raster
        - original Landsat scene count
        - unique acquisition-date count
    """

    start_date = f"{year}-06-01"
    end_date = f"{year}-10-01"

    # 1. Retrieve Landsat 8 scenes for Highline and this season
    scenes = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterBounds(highline.geometry())
        .filterDate(start_date, end_date)
    )

    # 2. Apply Step 3 preprocessing
    processed = scenes.map(preprocess_landsat8)

    # 3. Calculate NBR for every valid pixel observation
    nbr_scenes = processed.map(calculate_nbr)

    # 4. Find unique calendar acquisition dates
    acquisition_dates = (
        ee.List(nbr_scenes.aggregate_array("system:time_start"))
        .map(lambda t: ee.Date(t).format("YYYY-MM-dd"))
        .distinct()
        .sort()
    )

    # 5. Combine multiple scene products from the same date
    def combine_date(date_string):
        date = ee.Date.parse(
            "YYYY-MM-dd",
            ee.String(date_string)
        )

        same_day = nbr_scenes.filterDate(
            date,
            date.advance(1, "day")
        )

        daily_nbr = (
            same_day
            .median()
            .rename("NBR")
        )

        return ee.Image(daily_nbr).set({
            "date": date.format("YYYY-MM-dd"),
            "system:time_start": date.millis(),
            "source_scene_count": same_day.size(),
        })

    daily_nbr = ee.ImageCollection.fromImages(
        acquisition_dates.map(combine_date)
    )

    # 6. Temporal median across valid acquisition dates
    seasonal_nbr = (
        daily_nbr
        .median()
        .rename("NBR")
    )

    # 7. Count valid acquisition dates at every 30 m location
    observation_count = (
        daily_nbr
        .count()
        .rename("valid_obs_count")
    )

    return {
        "nbr": seasonal_nbr,
        "obs_count": observation_count,
        "scene_count": scenes.size(),
        "date_count": daily_nbr.size(),
    }

### Step 6.2 — Build Seasonal NBR for All Required Years

The reusable seasonal-NBR function is applied to the five years needed for the Highline proof of concept:

- **2014**
- **2015**
- **2016**
- **2018**
- **2022**

The first three years represent the pre-fire period.

Highline burned in **2017**, so:

- **2018** is the first complete post-fire growing season (`NBR_1`)
- **2022** is the fifth complete post-fire growing season (`NBR_5`)

For each year, we also inspect:

- the number of original Landsat scene products
- the number of unique acquisition dates after same-date scene products are combined

Results:

- 2014: 24 scenes → 16 unique dates
- 2015: 23 scenes → 15 unique dates
- 2016: 22 scenes → 14 unique dates
- 2018: 23 scenes → 15 unique dates
- 2022: 23 scenes → 15 unique dates

All five required years therefore have substantial Landsat coverage during the standardized June–September window.

In [28]:
years = [2014, 2015, 2016, 2018, 2022]

seasonal_results = {
    year: build_seasonal_nbr(year)
    for year in years
}

for year in years:
    print(
        year,
        "| Scene products:",
        seasonal_results[year]["scene_count"].getInfo(),
        "| Unique dates:",
        seasonal_results[year]["date_count"].getInfo(),
    )

2014 | Scene products: 24 | Unique dates: 16
2015 | Scene products: 23 | Unique dates: 15
2016 | Scene products: 22 | Unique dates: 14
2018 | Scene products: 23 | Unique dates: 15
2022 | Scene products: 23 | Unique dates: 15


### Step 6.3 — Check Annual Seasonal NBR and Observation Coverage

Before combining any years, each annual seasonal NBR raster is inspected independently.

For every year, we calculate:

- NBR 1st percentile
- NBR median
- NBR 99th percentile
- valid-observation-count 1st percentile
- valid-observation-count median
- valid-observation-count 99th percentile

The observation count represents how many valid Landsat acquisition dates contributed to a location's seasonal NBR estimate.

The broad spatial medians were approximately:

- **2014:** `0.293`
- **2015:** `0.301`
- **2016:** `0.293`
- **2018:** `0.004`
- **2022:** `0.098`

The three pre-fire years are very consistent.

The broad pattern is also physically sensible:

`stable pre-fire NBR`

→ **2017 wildfire**

→ `large decline in 2018`

→ `partial increase by 2022`

This is encouraging for the proof of concept, although these are whole-fire spatial summaries rather than individual pixel or final patch trajectories.

The pre-fire years also have somewhat fewer usable observations per pixel than some later years, which means a final minimum valid-observation threshold will need to be chosen later during data-quality analysis.

In [29]:
for year in years:

    nbr_image = seasonal_results[year]["nbr"]
    obs_image = seasonal_results[year]["obs_count"]

    nbr_stats = nbr_image.reduceRegion(
        reducer=ee.Reducer.percentile([1, 50, 99]),
        geometry=highline.geometry(),
        scale=30,
        bestEffort=True,
        maxPixels=1e8,
    )

    obs_stats = obs_image.reduceRegion(
        reducer=ee.Reducer.percentile([1, 50, 99]),
        geometry=highline.geometry(),
        scale=30,
        bestEffort=True,
        maxPixels=1e8,
    )

    print(f"\n{year}")
    print("NBR:", nbr_stats.getInfo())
    print("Valid observations:", obs_stats.getInfo())


2014
NBR: {'NBR_p1': -0.011565019314547849, 'NBR_p50': 0.29299114419652716, 'NBR_p99': 0.6132265189990735}
Valid observations: {'valid_obs_count_p1': 2, 'valid_obs_count_p50': 4, 'valid_obs_count_p99': 9.999999999999995}

2015
NBR: {'NBR_p1': -0.027133764200722853, 'NBR_p50': 0.3008045038281848, 'NBR_p99': 0.613115711898917}
Valid observations: {'valid_obs_count_p1': 2.874116198548117, 'valid_obs_count_p50': 4, 'valid_obs_count_p99': 9.000000000000002}

2016
NBR: {'NBR_p1': -0.01953200756536107, 'NBR_p50': 0.29300876234906004, 'NBR_p99': 0.5975189168454111}
Valid observations: {'valid_obs_count_p1': 3, 'valid_obs_count_p50': 6.000000000000013, 'valid_obs_count_p99': 12}

2018
NBR: {'NBR_p1': -0.45696043107787226, 'NBR_p50': 0.003924816483028613, 'NBR_p99': 0.5507525969881503}
Valid observations: {'valid_obs_count_p1': 5.000000000000002, 'valid_obs_count_p50': 5.9999999999999725, 'valid_obs_count_p99': 12.00000000000002}

2022
NBR: {'NBR_p1': -0.16796024035756252, 'NBR_p50': 0.09767830

### Step 6.4 — Create the Core Landsat Variables

The three pre-fire seasonal rasters from:

- 2014
- 2015
- 2016

are combined using a **pixel-wise median** to create the preliminary pre-fire baseline:

`NBR_pre`

For each 30 m location, the median of its available 2014, 2015, and 2016 seasonal NBR values is used.

The core Landsat variables for the Highline proof of concept are then defined as:

- `NBR_pre` → pre-fire baseline from 2014–2016
- `NBR_1` → 2018 seasonal NBR
- `NBR_5` → 2022 seasonal NBR

`NBR_5` will ultimately be the primary prediction target in the modeling dataset.

A separate raster is also created:

`valid_pre_year_count`

This records how many of the three pre-fire years contribute a valid seasonal NBR value at each location.

Possible values are conceptually:

- `0` → no valid pre-fire years
- `1` → one valid year
- `2` → two valid years
- `3` → all three valid years

In [30]:
# Build the preliminary pre-fire NBR baseline

pre_year_images = ee.ImageCollection.fromImages([
    seasonal_results[2014]["nbr"],
    seasonal_results[2015]["nbr"],
    seasonal_results[2016]["nbr"],
])

nbr_pre = (
    pre_year_images
    .median()
    .rename("NBR_pre")
)

# Count how many of the three pre-fire years contribute
# a valid seasonal NBR value at each 30 m location
nbr_pre_year_count = (
    pre_year_images
    .count()
    .rename("valid_pre_year_count")
)

# Season 1 and Season 5
nbr_1 = seasonal_results[2018]["nbr"].rename("NBR_1")
nbr_5 = seasonal_results[2022]["nbr"].rename("NBR_5")

print("Pre-fire band:", nbr_pre.bandNames().getInfo())
print("Season 1 band:", nbr_1.bandNames().getInfo())
print("Season 5 band:", nbr_5.bandNames().getInfo())
print(
    "Pre-fire year-count band:",
    nbr_pre_year_count.bandNames().getInfo()
)

Pre-fire band: ['NBR_pre']
Season 1 band: ['NBR_1']
Season 5 band: ['NBR_5']
Pre-fire year-count band: ['valid_pre_year_count']


### Step 6.5 — Preliminary `NBR_pre` and Pre-Fire Coverage Check

The resulting `NBR_pre` raster is summarized using its 1st, 50th, and 99th percentiles.

Results:

- p1: approximately `-0.012`
- median: approximately `0.293`
- p99: approximately `0.605`

This is consistent with the strong agreement among the individual 2014, 2015, and 2016 seasonal rasters.

We also inspect `valid_pre_year_count` to determine how many pre-fire years contribute at each location.

The initial histogram returned only the value:

`3`

suggesting that every location represented in the count raster has valid NBR information from all three pre-fire years.

However, the histogram value associated with `3` was approximately **599,529**, which appeared inconsistent with Highline's physical area and the roughly **420,000 full 30 m pixel-equivalents** implied by that area.

Because of this discrepancy, the histogram result should **not yet be interpreted as a literal number of Landsat pixels**.

This triggered the subsequent projection and area-based diagnostics in Steps 6A–6C before finalizing the pre-fire coverage interpretation.

In [31]:
pre_stats = nbr_pre.reduceRegion(
    reducer=ee.Reducer.percentile([1, 50, 99]),
    geometry=highline.geometry(),
    scale=30,
    bestEffort=True,
    maxPixels=1e8,
)

pre_year_count_stats = nbr_pre_year_count.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=highline.geometry(),
    scale=30,
    bestEffort=True,
    maxPixels=1e8,
)

print("NBR_pre statistics:")
print(pre_stats.getInfo())

print("\nNumber of valid pre-fire years per pixel:")
print(pre_year_count_stats.getInfo())

NBR_pre statistics:
{'NBR_pre_p1': -0.011600656647666201, 'NBR_pre_p50': 0.293004814642944, 'NBR_pre_p99': 0.6053076829980839}

Number of valid pre-fire years per pixel:
{'valid_pre_year_count': {'3': 599528.5333333298}}


## Step 6A — Investigating the Pre-Fire Pixel-Count Discrepancy

When we summarized `valid_pre_year_count`, the histogram returned approximately **599,529** for the value `3`, which initially appeared to suggest that roughly 599,529 Landsat pixels had valid pre-fire NBR in all three years.

However, Highline's MTBS-reported area is about **93,699 acres**. Since one Landsat optical pixel is approximately **30 m × 30 m = 900 m²**, the fire should correspond to only about **420,000 full 30 m pixel-equivalents**.

Because ~599,000 is substantially larger than ~420,000, we should not assume that the histogram output represents a literal count of distinct 30 m pixels.

Before interpreting the pre-fire coverage results, we therefore check:

1. the area reported by MTBS,
2. the area of the Highline geometry in Earth Engine,
3. the number of 30 m pixel-equivalents implied by that area.

This helps determine whether the discrepancy comes from the fire geometry itself or from how the raster reduction/histogram is being calculated.

In [32]:
# MTBS-reported area in acres
mtbs_acres = ee.Number(highline.get("BurnBndAc"))

# Earth Engine geometry area in square meters
geometry_area_m2 = highline.geometry().area(maxError=1)

# Convert geometry area to acres
geometry_acres = geometry_area_m2.divide(4046.8564224)

# One Landsat optical pixel is approximately 30 m x 30 m = 900 m²
pixel_area_m2 = 30 * 30

# Full-pixel-equivalent count implied by the geometry area
pixel_equivalents = geometry_area_m2.divide(pixel_area_m2)

print("MTBS reported acres:", mtbs_acres.getInfo())
print("Geometry area acres:", geometry_acres.getInfo())
print("Geometry area m²:", geometry_area_m2.getInfo())
print("30 m pixel-equivalents:", pixel_equivalents.getInfo())

MTBS reported acres: 93699
Geometry area acres: 93482.92740619012
Geometry area m²: 378311985.15849346
30 m pixel-equivalents: 420346.65017610387


### Step 6A.2 — Investigating the Raster Projection Behind the Pixel-Count Discrepancy

The previous histogram appeared to show approximately **599,529 pixels** with valid NBR in all three pre-fire years.

However, Highline's area is only about **378.3 million m²**, which corresponds to approximately:

`378,300,000 / 900 ≈ 420,000`

full **30 m × 30 m Landsat pixel-equivalents**.

Because ~599,000 is much larger than ~420,000, we checked whether the seasonal NBR composites were still using Landsat's native 30 m grid.

### What Is a Projection?

A raster projection defines how locations on the curved Earth are represented on a flat grid.

The original Landsat scene uses:

`EPSG:32611`

This is a **UTM projection** appropriate for the Highline region.

Its transform contains:

`30` and `-30`

which indicates that the raster grid is based on approximately **30 m × 30 m pixels**.

The original Landsat output was:

`EPSG:32611`

with a 30 m grid.

### What Happened to the Seasonal Composite?

The 2018 seasonal NBR raster and `NBR_pre` instead reported:

`EPSG:4326`

with a transform of:

`[1, 0, 0, 0, 1, 0]`

`EPSG:4326` is the standard latitude/longitude coordinate system, where coordinates are represented in **degrees rather than meters**.

A degree does not represent the same physical distance everywhere on Earth.

For example, around Highline's latitude, roughly in the mid-40s °N, one degree of longitude is substantially narrower than one degree at the equator.

Therefore, when Earth Engine performs a reduction using:

`scale = 30`

on an image whose default projection is `EPSG:4326`, Earth Engine must internally determine how that requested 30 m scale maps onto the latitude/longitude grid.

The resulting reducer samples or weights should therefore **not automatically be interpreted as literal native 30 m Landsat pixels**.

### Why Did the Composite Lose the Original Landsat Projection?

The original Landsat images have a native UTM grid.

However, operations such as taking a median across an `ImageCollection` containing multiple images can produce a composite whose default projection falls back to Earth Engine's generic latitude/longitude projection.

This is what occurred for:

- the 2018 seasonal NBR composite
- `NBR_pre`

### Interpretation of the Discrepancy

The projection check indicates that the earlier histogram result of approximately:

`599,529`

should **not** be interpreted as:

> "Highline contains 599,529 actual 30 m Landsat pixels."

The more physically meaningful reference is the fire's geographic area:

- MTBS reported area: approximately **93,699 acres**
- Earth Engine geometry area: approximately **93,483 acres**
- geometry area: approximately **378.3 million m²**
- equivalent number of full 30 m × 30 m pixels: approximately **420,347**

The MTBS area and Earth Engine geometry agree closely, so the fire boundary itself is behaving sensibly.

The apparent ~599k value arose because the histogram reduction was performed on a composite with a generic `EPSG:4326` projection and was mistakenly interpreted as a literal native-pixel count.

In [33]:
# Inspect source vs. composite raster projections

source_scene = ee.Image(landsat_s1.first())

print("Original Landsat SR_B5 projection:")
print(source_scene.select("SR_B5").projection().getInfo())

print("\n2018 seasonal NBR projection:")
print(seasonal_results[2018]["nbr"].projection().getInfo())

print("\nNBR_pre projection:")
print(nbr_pre.projection().getInfo())

Original Landsat SR_B5 projection:
{'type': 'Projection', 'crs': 'EPSG:32611', 'transform': [30, 0, 569685, 0, -30, 5216415]}

2018 seasonal NBR projection:
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}

NBR_pre projection:
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [1, 0, 0, 0, 1, 0]}


### Step 6B — Pre-Fire Year Coverage Check

After building `NBR_pre` from the 2014, 2015, and 2016 seasonal NBR rasters, we checked how many pre-fire years contributed a valid seasonal NBR value at each location.

The previous histogram output was not treated as a literal pixel count because the composite raster had fallen back to a generic `EPSG:4326` projection.

Instead, we used an **area-based diagnostic** and explicitly treated masked count locations as zero.

Result:

- 0 valid pre-fire years → no area detected
- 1 valid pre-fire year → no area detected
- 2 valid pre-fire years → no area detected
- 3 valid pre-fire years → approximately **379.18 million m²**

This means essentially the entire Highline perimeter has a valid seasonal NBR value in **all three pre-fire years: 2014, 2015, and 2016**.

This does not mean every year had the same number of usable Landsat dates; it only means each location successfully produced a seasonal NBR value in all three years.

In [34]:
# Convert masked locations to explicit 0 for this diagnostic
pre_year_count_complete = (
    nbr_pre_year_count
    .unmask(0)
    .clip(highline.geometry())
)

# Add pixel area in square meters
area_by_count_image = ee.Image.pixelArea().addBands(
    pre_year_count_complete
)

# Sum geographic area for each count category: 0, 1, 2, 3
area_by_count = area_by_count_image.reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,
        groupName="valid_pre_years"
    ),
    geometry=highline.geometry(),
    scale=30,
    maxPixels=1e8,
)

print(area_by_count.getInfo())

{'groups': [{'sum': 379177897.2881001, 'valid_pre_years': 3}]}


In [35]:
coverage_area_m2 = 379177897.2881001

coverage_acres = coverage_area_m2 / 4046.8564224
geometry_area_m2_value = 378311985.11378723

print("Area with all 3 valid pre-fire years (acres):", coverage_acres)
print(
    "Raster-area / vector-area ratio:",
    coverage_area_m2 / geometry_area_m2_value
)

Area with all 3 valid pre-fire years (acres): 93696.89895329361
Raster-area / vector-area ratio: 1.0022888838006347


### Step 6C — Whole-Fire Coverage for `NBR_1` and `NBR_5`

We also checked whether any part of the Highline fire perimeter was missing a usable seasonal NBR value for:

- `NBR_1` — Season 1, 2018
- `NBR_5` — Season 5, 2022

For each raster, locations with a valid seasonal NBR were coded as `1`, while missing or masked locations were coded as `0`. Geographic area was then summed by coverage category.

Results:

- `NBR_1`: approximately **379.18 million m²** with valid coverage
- `NBR_5`: approximately **379.18 million m²** with valid coverage
- no zero-coverage area was detected for either raster

Therefore, the current Highline proof of concept has complete seasonal NBR coverage for both Season 1 and Season 5 across the evaluated fire perimeter.

In [36]:
def coverage_area_stats(image, band_name):
    # 1 where the seasonal NBR exists, 0 where it is masked
    valid_mask = (
        image.select(band_name)
        .mask()
        .unmask(0)
        .rename("valid")
        .clip(highline.geometry())
    )

    # Convert valid/missing coverage into geographic area
    area_image = ee.Image.pixelArea().addBands(valid_mask)

    grouped = area_image.reduceRegion(
        reducer=ee.Reducer.sum().group(
            groupField=1,
            groupName="valid"
        ),
        geometry=highline.geometry(),
        scale=30,
        maxPixels=1e8,
    )

    return grouped


nbr1_coverage = coverage_area_stats(nbr_1, "NBR_1")
nbr5_coverage = coverage_area_stats(nbr_5, "NBR_5")

print("NBR_1 coverage:")
print(nbr1_coverage.getInfo())

print("\nNBR_5 coverage:")
print(nbr5_coverage.getInfo())

NBR_1 coverage:
{'groups': [{'sum': 379177897.2881001, 'valid': 1}]}

NBR_5 coverage:
{'groups': [{'sum': 379177897.2881001, 'valid': 1}]}


### Step 7A — Visual Validation of Core NBR Rasters

`NBR_pre`, `NBR_1`, and `NBR_5` were mapped over the Highline perimeter using the same NBR visualization scale (`-0.5` to `0.7`) so colors are directly comparable across time.

The maps show the expected broad temporal pattern:

- **Pre-fire:** predominantly higher NBR across Highline
- **Season 1 (2018):** substantial shift toward lower NBR, especially in the southern portion of the fire
- **Season 5 (2022):** noticeable increase relative to Season 1, indicating partial spectral vegetation recovery

No obvious scene seams, coverage gaps, or major compositing artifacts were visible.

The colors represent **Landsat-derived NBR values only**. LANDFIRE vegetation information has not yet been applied; MTBS is currently used only for the Highline fire perimeter in this visualization.

In [42]:
import geemap

# Step 7A: Visualize NBR_pre, NBR_1, and NBR_5

# Clip only for display
nbr_pre_display = nbr_pre.clip(highline.geometry())
nbr_1_display = nbr_1.clip(highline.geometry())
nbr_5_display = nbr_5.clip(highline.geometry())

# Use the SAME visualization scale for all three rasters
nbr_vis = {
    "min": -0.5,
    "max": 0.7,
    "palette": [
        "8c510a",
        "d8b365",
        "f6e8c3",
        "c7eae5",
        "5ab4ac",
        "01665e",
    ],
}

# Explicitly cast Highline into a FeatureCollection
# so that Earth Engine's .style() method is available
highline_outline = ee.FeatureCollection([
    ee.Feature(highline)
])

Map = geemap.Map()

Map.centerObject(highline, 10)

Map.addLayer(
    nbr_pre_display,
    nbr_vis,
    "NBR pre-fire (2014–2016)"
)

Map.addLayer(
    nbr_1_display,
    nbr_vis,
    "NBR Season 1 (2018)"
)

Map.addLayer(
    nbr_5_display,
    nbr_vis,
    "NBR Season 5 (2022)"
)

Map.addLayer(
    highline_outline.style(
        color="000000",
        fillColor="00000000",
        width=2
    ),
    {},
    "Highline perimeter"
)

Map

Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

### Step 7B — Computational Challenge and Engineering Lesson

The first attempts to calculate and visualize NBR change directly from the in-memory Earth Engine objects failed with:

`User memory limit exceeded`

The subtraction itself was not computationally difficult. The issue was that `NBR_pre`, `NBR_1`, and `NBR_5` were still lazily evaluated Earth Engine computation graphs rather than stored rasters.

For example, evaluating `NBR_pre - NBR_1` required Earth Engine to reconstruct the underlying Landsat processing for 2014, 2015, 2016, and 2018, including QA masking, reflectance scaling, NBR calculation, same-date compositing, and seasonal medians before performing the subtraction.

The following approaches were attempted:

1. **Whole-fire 30 m percentile reduction**
   - Failed with a memory-limit error because the derived change raster required several seasonal Landsat pipelines to be evaluated simultaneously.

2. **Clipping to Highline and increasing `tileScale`**
   - Still failed because smaller processing tiles did not remove the large upstream computation graph.

3. **Sampling approximately 5,000 locations**
   - Also failed because Earth Engine still had to evaluate the complex multi-season inputs before generating the sample.

4. **Direct visualization of the derived change raster**
   - Failed during map generation for the same reason: the map request still depended on reconstructing multiple seasonal pipelines.

The solution was to **materialize the completed Step 6 outputs** as Earth Engine assets:

- `highline_nbr_pre`
- `highline_nbr_1`
- `highline_nbr_5`

An initial export attempt failed because the Cloud project's Earth Engine asset root did not yet exist. The asset folder was created using the Earth Engine CLI with the project explicitly specified, after which all three exports completed successfully.

The materialized rasters were written to the same explicit 30 m `EPSG:32611` grid. Once reloaded, the computation became shallow:

`saved seasonal rasters → subtraction → statistics / visualization`

The original full-resolution percentile calculation and change maps then completed successfully.

**Engineering lesson:** when a lazy geospatial computation graph becomes deep or repeatedly reused, materializing a meaningful intermediate product can greatly simplify downstream computation. This pattern will be important when the full multi-fire pipeline is scaled and partitioned in Phase 3.

In [75]:

# Materialize the three core Highline NBR rasters only if they do not already exist

asset_root = "projects/ee-wildfire-recovery-vasant/assets"

asset_ids = {
    "NBR_pre": f"{asset_root}/highline_nbr_pre",
    "NBR_1": f"{asset_root}/highline_nbr_1",
    "NBR_5": f"{asset_root}/highline_nbr_5",
}

images_to_export = {
    "NBR_pre": nbr_pre,
    "NBR_1": nbr_1,
    "NBR_5": nbr_5,
}

# Holds tasks started during THIS notebook run
export_tasks = {}

for name, asset_id in asset_ids.items():

    try:
        # Asset already exists → reuse it
        ee.data.getAsset(asset_id)
        print(f"{name}: EXISTS — reusing saved asset")

    except ee.EEException as e:
        # Only treat "not found" as a reason to create the asset
        if "not found" not in str(e).lower():
            raise

        task = ee.batch.Export.image.toAsset(
            image=images_to_export[name].clip(highline.geometry()),
            description=f"highline_{name.lower()}",
            assetId=asset_id,
            region=highline.geometry(),
            scale=30,
            crs="EPSG:32611",
            maxPixels=1e8,
        )

        task.start()
        export_tasks[name] = task

        print(f"{name}: EXPORT STARTED")

NBR_pre: EXISTS — reusing saved asset
NBR_1: EXISTS — reusing saved asset
NBR_5: EXISTS — reusing saved asset


In [76]:
# Check materialization status

for name, asset_id in asset_ids.items():

    # If this notebook run started an export, report its task status
    if name in export_tasks:
        status = export_tasks[name].status()
        state = status["state"]

        print(f"{name}: {state}")

        if state == "FAILED":
            print("  Error:", status.get("error_message"))

    else:
        # No task was started because the asset already existed
        try:
            ee.data.getAsset(asset_id)
            print(f"{name}: EXISTS / READY TO USE")
        except ee.EEException:
            print(f"{name}: MISSING")

NBR_pre: EXISTS / READY TO USE
NBR_1: EXISTS / READY TO USE
NBR_5: EXISTS / READY TO USE


### Step 7B — Spatial NBR Change Analysis

To make the temporal changes explicit, two pixel-level change rasters were calculated from the materialized 30 m Landsat outputs:

- `NBR_drop = NBR_pre - NBR_1`
  - positive values indicate a decline from the pre-fire baseline to Season 1
- `NBR_gain = NBR_5 - NBR_1`
  - positive values indicate an increase from Season 1 to Season 5

The saved `NBR_pre`, `NBR_1`, and `NBR_5` rasters were confirmed to share the same 30 m `EPSG:32611` grid and transform before subtraction.


#### NBR Drop: Pre-Fire → Season 1

Percentiles across Highline:

- p1: `-0.098`
- p10: `0.012`
- median: `0.262`
- p90: `0.629`
- p99: `0.910`

The typical location experienced a substantial NBR decline after the fire, with very large declines in the most strongly affected areas.

#### NBR Gain: Season 1 → Season 5

Percentiles across Highline:

- p1: `-0.121`
- p10: `-0.019`
- median: `0.098`
- p90: `0.285`
- p99: `0.457`

Most of Highline shows positive NBR change between Season 1 and Season 5, but the amount of recovery varies substantially across space.

In [77]:
# Step 7B.1: Reload materialized NBR rasters and quantify spatial change across Highline

asset_root = "projects/ee-wildfire-recovery-vasant/assets"

# Reload the finished Step 6 outputs as simple stored rasters
nbr_pre_saved = ee.Image(f"{asset_root}/highline_nbr_pre")
nbr_1_saved = ee.Image(f"{asset_root}/highline_nbr_1")
nbr_5_saved = ee.Image(f"{asset_root}/highline_nbr_5")

# Confirm that the exported rasters now have the explicit 30 m UTM projection we requested
print("NBR_pre projection:")
print(nbr_pre_saved.projection().getInfo())

print("\nNBR_1 projection:")
print(nbr_1_saved.projection().getInfo())

print("\nNBR_5 projection:")
print(nbr_5_saved.projection().getInfo())

# Calculate the two change rasters

# Positive = NBR declined from pre-fire to Season 1
nbr_drop_saved = (
    nbr_pre_saved
    .subtract(nbr_1_saved)
    .rename("NBR_drop")
)

# Positive = NBR increased from Season 1 to Season 5
nbr_gain_saved = (
    nbr_5_saved
    .subtract(nbr_1_saved)
    .rename("NBR_gain")
)


# Calculate whole-fire percentiles at the native 30 m scale

drop_stats = nbr_drop_saved.reduceRegion(
    reducer=ee.Reducer.percentile([1, 10, 50, 90, 99]),
    geometry=highline.geometry(),
    scale=30,
    crs="EPSG:32611",
    maxPixels=1e8,
    tileScale=4,
)

gain_stats = nbr_gain_saved.reduceRegion(
    reducer=ee.Reducer.percentile([1, 10, 50, 90, 99]),
    geometry=highline.geometry(),
    scale=30,
    crs="EPSG:32611",
    maxPixels=1e8,
    tileScale=4,
)

print("\nNBR drop: NBR_pre - NBR_1")
print(drop_stats.getInfo())

print("\nNBR gain: NBR_5 - NBR_1")
print(gain_stats.getInfo())

NBR_pre projection:
{'type': 'Projection', 'crs': 'EPSG:32611', 'transform': [30, 0, 633600, 0, -30, 5042430]}

NBR_1 projection:
{'type': 'Projection', 'crs': 'EPSG:32611', 'transform': [30, 0, 633600, 0, -30, 5042430]}

NBR_5 projection:
{'type': 'Projection', 'crs': 'EPSG:32611', 'transform': [30, 0, 633600, 0, -30, 5042430]}

NBR drop: NBR_pre - NBR_1
{'NBR_drop_p1': -0.09750911166934213, 'NBR_drop_p10': 0.011694244333383511, 'NBR_drop_p50': 0.2617718073473328, 'NBR_drop_p90': 0.628981692603124, 'NBR_drop_p99': 0.9101428460041576}

NBR gain: NBR_5 - NBR_1
{'NBR_gain_p1': -0.12111797258982868, 'NBR_gain_p10': -0.019423314550532138, 'NBR_gain_p50': 0.09761653670476443, 'NBR_gain_p90': 0.28517376625241886, 'NBR_gain_p99': 0.45683588405112563}


In [78]:
# Step 7B.2: Visualize spatial NBR change using the materialized Step 6 rasters

# Symmetric visualization ranges keep zero at the center of each diverging color scale.

drop_vis = {
    "min": -0.9,
    "max": 0.9,
    "palette": [
        "2166ac",
        "67a9cf",
        "d1e5f0",
        "f7f7f7",
        "fddbc7",
        "ef8a62",
        "b2182b",
    ],
}

gain_vis = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "b2182b",
        "ef8a62",
        "fddbc7",
        "f7f7f7",
        "d1e5f0",
        "67a9cf",
        "2166ac",
    ],
}

highline_outline = ee.FeatureCollection([
    ee.Feature(highline)
])

Map_7b = geemap.Map()
Map_7b.centerObject(highline, 10)

Map_7b.addLayer(
    nbr_drop_saved,
    drop_vis,
    "NBR drop: pre-fire - Season 1"
)

Map_7b.addLayer(
    nbr_gain_saved,
    gain_vis,
    "NBR gain: Season 5 - Season 1"
)

Map_7b.addLayer(
    highline_outline.style(
        color="000000",
        fillColor="00000000",
        width=2
    ),
    {},
    "Highline perimeter"
)

Map_7b

Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

The maps reinforce this pattern: the strongest initial declines are concentrated particularly in the southern portion of the fire, while many of those areas also show substantial subsequent NBR gain.

Overall, the Highline POC shows a clear and spatially heterogeneous pattern of **major post-fire vegetation decline followed by partial spectral recovery by Season 5**. Positive Season 1-to-Season 5 gain should not be interpreted as full return to the pre-fire condition.

### Step 7C - Save important views as images instead of just relying on the map

In [81]:
from pathlib import Path
from urllib.request import urlretrieve

# Resolve repo root and output folder
cwd = Path.cwd()
repo_root = cwd if (cwd / "pyproject.toml").exists() else cwd.parent

landsat_fig_dir = repo_root / "outputs" / "figures" / "landsat"
landsat_fig_dir.mkdir(parents=True, exist_ok=True)

# Load the saved/materialized rasters from Earth Engine assets
asset_root = "projects/ee-wildfire-recovery-vasant/assets"

nbr_pre_saved = ee.Image(f"{asset_root}/highline_nbr_pre")
nbr_1_saved   = ee.Image(f"{asset_root}/highline_nbr_1")
nbr_5_saved   = ee.Image(f"{asset_root}/highline_nbr_5")

nbr_drop_saved = nbr_pre_saved.subtract(nbr_1_saved).rename("NBR_drop")
nbr_gain_saved = nbr_5_saved.subtract(nbr_1_saved).rename("NBR_gain")

# Geometry styling
outline = (
    ee.Image()
    .byte()
    .paint(highline_outline, 1, 2)
    .selfMask()
    .visualize(palette=["000000"])
)

# Visualization configs
nbr_vis = {
    "min": -0.5,
    "max": 0.7,
    "palette": [
        "8c510a",
        "d8b365",
        "f6e8c3",
        "c7eae5",
        "5ab4ac",
        "01665e",
    ],
}

drop_vis = {
    "min": -0.9,
    "max": 0.9,
    "palette": [
        "2166ac",
        "67a9cf",
        "d1e5f0",
        "f7f7f7",
        "fddbc7",
        "ef8a62",
        "b2182b",
    ],
}

gain_vis = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "b2182b",
        "ef8a62",
        "fddbc7",
        "f7f7f7",
        "d1e5f0",
        "67a9cf",
        "2166ac",
    ],
}

# Reusable save helper
def save_ee_png(image, vis_params, filename, dimensions=1600):
    output_path = landsat_fig_dir / filename

    rgb = image.clip(highline.geometry()).visualize(**vis_params)
    figure_image = rgb.blend(outline)

    thumb_url = figure_image.getThumbURL({
        "region": highline.geometry(),
        "dimensions": dimensions,
        "format": "png",
    })

    urlretrieve(thumb_url, output_path)
    print("Saved:", output_path)

# Save all key Landsat figures
save_ee_png(nbr_pre_saved, nbr_vis, "highline_nbr_pre.png")
save_ee_png(nbr_1_saved,   nbr_vis, "highline_nbr_1.png")
save_ee_png(nbr_5_saved,   nbr_vis, "highline_nbr_5.png")
save_ee_png(nbr_drop_saved, drop_vis, "highline_nbr_drop.png")
save_ee_png(nbr_gain_saved, gain_vis, "highline_nbr_gain.png")

Saved: outputs\figures\landsat\highline_nbr_pre.png
Saved: outputs\figures\landsat\highline_nbr_1.png
Saved: outputs\figures\landsat\highline_nbr_5.png
Saved: outputs\figures\landsat\highline_nbr_drop.png
Saved: outputs\figures\landsat\highline_nbr_gain.png
